In [13]:
import sys
sys.path.insert(0, '..')

import numpy as np
import importlib
import lattice.coder as coder_mod
importlib.reload(coder_mod)

from lattice.coder import PathCoder, Step, Path
from core.tensor import Tensor

In [14]:
# ---------------------------------------------------------------------------
# Leg table: (x, y, temp) -> result
#   x  = query/state vector being transformed
#   y  = relation matrix (shared across all steps)
#   temp = softmax temperature (0.0 = hard Boolean)
#
# realize  (se) : C→E   Join(x_row, y.T)        — extent of concept x
# propagate(sd) : E→C   Join(x_row, y)           — image under Join
# abstract (pe) : E→C   Residuate(y, x_col)      — intent of entity x
# support  (pd) : C→E   Residuate(y.T, x_col)    — preimage under Residuate
# ---------------------------------------------------------------------------
tensor = Tensor()

legs = [
    lambda x, y, t: tensor.Join(x.reshape(1, -1), y.T, t).squeeze(),   # realize (se)
    lambda x, y, t: tensor.Join(x.reshape(1, -1), y,   t).squeeze(),   # propagate (sd)
    lambda x, y, t: tensor.Residuate(y,   x.reshape(-1, 1), t).squeeze(),  # abstract (pe)
    lambda x, y, t: tensor.Residuate(y.T, x.reshape(-1, 1), t).squeeze(),  # support (pd)
]

In [15]:
# ---------------------------------------------------------------------------
# Relation matrix: 6 entities × 4 concepts (Boolean)
# ---------------------------------------------------------------------------
rel = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [1.0, 1.0, 0.0, 0.0],
    [0.0, 1.0, 1.0, 0.0],
    [0.0, 0.0, 1.0, 1.0],
    [0.0, 0.0, 0.0, 1.0],
    [1.0, 0.0, 1.0, 0.0],
], dtype=float)

# query vectors
q1 = np.array([1.0, 0.0, 0.0, 0.0], dtype=float)   # concept-space (C, dim=4)
q2 = np.array([0.0, 1.0, 1.0, 0.0], dtype=float)   # concept-space
x1 = np.array([1.0, 1.0, 0.0, 0.0, 0.0, 1.0], dtype=float)  # entity-space (E, dim=6)

temp = 0.0
print("rel:", rel.shape, "  q1:", q1.shape, "  x1:", x1.shape)

rel: (6, 4)   q1: (4,)   x1: (6,)


In [16]:
coder = PathCoder(legs)
print(coder.explain("se pe"))
print()
print(coder.explain("pe se"))

Path: se pe
Factorization: Σ.encode → Σ.decode → Δ → Π.encode → Π.decode
1. realize = Σ.encode  [Join(x, y.T)]
2. abstract = Π.encode  [Residuate(y, x)]

Path: pe se
Factorization: Σ.encode → Σ.decode → Δ → Π.encode → Π.decode
1. abstract = Π.encode  [Residuate(y, x)]
2. realize = Σ.encode  [Join(x, y.T)]


In [17]:
# ---------------------------------------------------------------------------
# Parsing and type-checking
# ---------------------------------------------------------------------------

# valid paths
print("parse 'se pe':", coder.parse("se pe"))
print("parse 'pe se':", coder.parse("pe se"))
print("parse 'sd pd sd pd':", coder.parse("sd pd sd pd"))  # same-pair folded to 2 steps

# type error: two C→E legs in a row
try:
    coder.parse("se sd")   # se:C→E, sd:E→C — valid
    coder.parse("se se")   # se:C→E, se:C→E — invalid (C≠E)
except TypeError as e:
    print("TypeError:", e)

parse 'se pe': Path(source='se pe', steps=(Step(name='realize', swap=False, same=False, trans=False), Step(name='abstract', swap=False, same=False, trans=False)), prog=None)
parse 'pe se': Path(source='pe se', steps=(Step(name='abstract', swap=False, same=False, trans=False), Step(name='realize', swap=False, same=False, trans=False)), prog=None)
parse 'sd pd sd pd': Path(source='sd pd sd pd', steps=(Step(name='propagate', swap=False, same=False, trans=False), Step(name='support', swap=False, same=False, trans=False)), prog=None)
TypeError: Type mismatch at step 1: 'realize' outputs 'E' but 'realize' expects 'C'


In [25]:
try:
    coder.parse("sd sd")   # se:C→E, sd:E→C — valid
except TypeError as e:
    print("TypeError:", e)

TypeError: Type mismatch at step 1: 'propagate' outputs 'C' but 'propagate' expects 'E'


In [18]:
# ---------------------------------------------------------------------------
# Modifier syntax: symmetry (swap x/y), diagonal (same=x,x), converse (trans y)
# ---------------------------------------------------------------------------

# symmetry swaps x and y passed to the leg
step_swap = coder.parse("se:symmetry").steps[0]
print("swap flag:", step_swap.swap, step_swap.same, step_swap.trans)

# diagonal: passes (x, x, t) — self-relation
step_diag = coder.parse("se:diagonal").steps[0]
print("diag flag:", step_diag.same)

# converse: transposes y before passing — useful for reverse traversal
step_conv = coder.parse("se:converse").steps[0]
print("conv flag:", step_conv.trans)

# combined
step_combo = coder.parse("se:diagonal:converse").steps[0]
print("combo flags:", step_combo.same, step_combo.trans)

swap flag: True False False
diag flag: True
conv flag: True
combo flags: True True


In [19]:
# ---------------------------------------------------------------------------
# run() and op(): execute paths with real tensors
# ---------------------------------------------------------------------------

# concept roundtrip: q1 -> realize -> abstract (se pe)
r_sepe = coder.run("se pe", q1, rel, temp)
print("se pe (concept roundtrip) q1:", r_sepe)

r_sepe2 = coder.run("se pe", q2, rel, temp)
print("se pe (concept roundtrip) q2:", r_sepe2)

# entity roundtrip: x1 -> abstract -> realize (pe se)
r_pese = coder.run("pe se", x1, rel, temp)
print("pe se (entity roundtrip)  x1:", r_pese)

# Attend: se sd
r_sesd = coder.run("se sd", q1, rel, temp)
print("se sd (Attend)            q1:", r_sesd)

# Recall: pe pd
r_pepd = coder.run("pe pd", x1, rel, temp)
print("pe pd (Recall)            x1:", r_pepd)

se pe (concept roundtrip) q1: [ 1.e+09 -1.e+09 -1.e+09 -1.e+09]
se pe (concept roundtrip) q2: [-1.e+09  1.e+09  1.e+09 -1.e+09]
pe se (entity roundtrip)  x1: [1. 1. 1. 1. 1. 1.]
se sd (Attend)            q1: [ 1.e+00  1.e+00  1.e+00 -1.e+09]
pe pd (Recall)            x1: [1.e+09 1.e+09 1.e+09 1.e+09 1.e+09 1.e+09]


In [20]:
# ---------------------------------------------------------------------------
# op(): returns a compiled callable (x, y, temp)
# ---------------------------------------------------------------------------

f_sepe = coder.op("se pe")
f_pese = coder.op("pe se")

print("op se pe (q1):", f_sepe(q1, rel, temp))
print("op pe se (x1):", f_pese(x1, rel, temp))

# cache: same spec returns identical object
assert coder.compile("se pe") is coder.compile("se pe")

op se pe (q1): [ 1.e+09 -1.e+09 -1.e+09 -1.e+09]
op pe se (x1): [1. 1. 1. 1. 1. 1.]


In [21]:
# ---------------------------------------------------------------------------
# trace(): step-by-step inspection
# ---------------------------------------------------------------------------

print("--- trace: se pe on q1 ---")
for step_name, info, shape, value in coder.trace("se pe", q1, rel, temp):
    print(f"  {step_name:12s}  {str(shape):12s}  {value}")

print()
print("--- trace: pe se on x1 ---")
for step_name, info, shape, value in coder.trace("pe se", x1, rel, temp):
    print(f"  {step_name:12s}  {str(shape):12s}  {value}")

--- trace: se pe on q1 ---
  input         (4,)          [1. 0. 0. 0.]
  realize       (6,)          [ 1.e+00  1.e+00 -1.e+09 -1.e+09 -1.e+09  1.e+00]
  abstract      (4,)          [ 1.e+09 -1.e+09 -1.e+09 -1.e+09]

--- trace: pe se on x1 ---
  input         (6,)          [1. 1. 0. 0. 0. 1.]
  abstract      (4,)          [1.e+09 1.e+09 1.e+09 1.e+09]
  realize       (6,)          [1. 1. 1. 1. 1. 1.]


In [22]:
# ---------------------------------------------------------------------------
# Idempotence: same-pair folding (algebraically guaranteed)
# (pd sd)^2 = pd sd,  (sd pd)^2 = sd pd,  (se pe)^2 = se pe,  (pe se)^2 = pe se
# ---------------------------------------------------------------------------

path4 = coder.parse("pd sd pd sd")
print("pd sd pd sd -> steps:", len(path4.steps), [s.name for s in path4.steps])

path4b = coder.parse("se pe se pe")
print("se pe se pe -> steps:", len(path4b.steps), [s.name for s in path4b.steps])

# Empirical pairs (se sd, pe pd) NOT folded by default
path_emp = coder.parse("se sd se sd")
print("se sd se sd (no fold) -> steps:", len(path_emp.steps))

# opt-in empirical folding via compile(..., fold_empirical=True)
path_emp_folded = coder.compile("se sd se sd", fold_empirical=True)
print("se sd se sd (fold_empirical=True) -> steps:", len(path_emp_folded.steps))

pd sd pd sd -> steps: 2 ['support', 'propagate']
se pe se pe -> steps: 2 ['realize', 'abstract']
se sd se sd (no fold) -> steps: 4
se sd se sd (fold_empirical=True) -> steps: 2


In [23]:
# ---------------------------------------------------------------------------
# Instance-level fold_empirical flag
# PathCoder(legs, fold_empirical=True) auto-folds empirical pairs on every compile
# ---------------------------------------------------------------------------

coder_fold = PathCoder(legs, fold_empirical=True)
path_auto = coder_fold.compile("se sd se sd")
print("instance fold_empirical=True, se sd se sd -> steps:", len(path_auto.steps))

path_auto2 = coder_fold.compile("pe pd pe pd")
print("instance fold_empirical=True, pe pd pe pd -> steps:", len(path_auto2.steps))

instance fold_empirical=True, se sd se sd -> steps: 2
instance fold_empirical=True, pe pd pe pd -> steps: 2


In [24]:
# ---------------------------------------------------------------------------
# Soft (finite temperature) behaviour
# At temp > 0 the semiring is Log-sum-exp, not Boolean max.
# A ⊕_T A = A + T*log(2) ≠ A  (non-idempotent addition at finite T)
# ---------------------------------------------------------------------------

temp_soft = 0.5
r_hard = coder.run("se pe", q1, rel, 0.0)
r_soft = coder.run("se pe", q1, rel, temp_soft)
print("se pe q1 hard (T=0.0):", r_hard)
print("se pe q1 soft (T=0.5):", r_soft)

se pe q1 hard (T=0.0): [ 1.e+09 -1.e+09 -1.e+09 -1.e+09]
se pe q1 soft (T=0.5): [ 1.04120265e-01 -1.00000000e+09 -1.00000000e+09 -1.00000000e+09]
